# 🚀 Linear Block Diffusion — Kaggle Training & Evaluation

**Architecture**: `LinearBlockDiffusionArchitecture` from `pymbbo` (~41M Parameters)  
**Hardware**: Kaggle Dual NVIDIA Tesla T4 GPUs (DataParallel + FP16 AMP)  
**Tokenizer**: HuggingFace `gpt2` (`vocab_size = 50257`)  
**Dataset**: HuggingFace `wikitext-2-raw-v1` (1024 tokens/sample)  

---

### 📊 Métricas de Evaluación
| Categoría | Métrica |
|---|---|
| ⚡ Entrenamiento | Tokens/s, ms/paso |
| 🚀 Inferencia | Tokens/s, ms/bloque |
| 💾 VRAM | Uso máximo GPU (GB) |
| 📉 Calidad | Perplejidad (PPL), Loss de Validación |
| 📝 Generación | Muestras de texto decodificadas con GPT-2 |

In [ ]:
# ── Cell 1: Instalación de dependencias ─────────────────────────────────
# Instalamos la última versión del repositorio con parches de rendimiento
!pip install -q git+https://github.com/bueormnew/pymbbo.git datasets transformers accelerate pandas

In [ ]:
# ── Cell 2: Diagnóstico de Hardware & Setup ─────────────────────────────
import os, sys, math, time, random
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print("=" * 75)
print("🖥️  KAGGLE DUAL T4 GPU — HARDWARE DIAGNOSTIC")
print("=" * 75)
print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")

NUM_GPUS = 0
if torch.cuda.is_available():
    NUM_GPUS = torch.cuda.device_count()
    for i in range(NUM_GPUS):
        props = torch.cuda.get_device_properties(i)
        print(f"  [GPU {i}] {props.name} | VRAM: {props.total_memory / 1e9:.2f} GB")
else:
    print("⚠️  CUDA no detectado. Activa GPU Accelerator en Kaggle Settings.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice seleccionado: {DEVICE}")

In [ ]:
# ── Cell 3: Patch de compatibilidad de typing & import de pymbbo ────────
import typing, builtins
for _name in ['Tuple', 'List', 'Dict', 'Optional', 'Union', 'Any', 'Callable']:
    if not hasattr(builtins, _name):
        setattr(builtins, _name, getattr(typing, _name))

from pymbbo.architectures.linear_block_diffusion import LinearBlockDiffusionArchitecture

print("✅ LinearBlockDiffusionArchitecture importada correctamente desde pymbbo.")

In [ ]:
# ── Cell 4: Dataset de HuggingFace + GPT-2 Tokenizer ───────────────────
from datasets import load_dataset
from transformers import AutoTokenizer

print("=" * 75)
print("🤗 DESCARGANDO DATASET & GPT-2 TOKENIZER")
print("=" * 75)

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
VOCAB_SIZE = tokenizer.vocab_size  # 50257
print(f"Tokenizer GPT-2 cargado: vocab_size = {VOCAB_SIZE:,}")

raw_dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
train_texts = [t for t in raw_dataset["train"]["text"] if len(t.strip()) > 50]
val_texts   = [t for t in raw_dataset["validation"]["text"] if len(t.strip()) > 50]
print(f"Textos filtrados: Train={len(train_texts):,} | Val={len(val_texts):,}")

# ── Hiperparámetros de datos ──
SEQ_LEN    = 1024   # tokens de target por sample
PROMPT_LEN = 64     # tokens de prompt por sample
TOTAL_LEN  = PROMPT_LEN + SEQ_LEN  # 1088

def tokenize_and_chunk(texts, max_samples=8000):
    """Tokeniza textos y genera pares (prompt, target) con ventana deslizante."""
    all_ids = []
    for text in texts:
        all_ids.extend(tokenizer.encode(text))
        if len(all_ids) >= max_samples * 512:
            break

    prompts, targets = [], []
    stride = 256
    for i in range(0, len(all_ids) - TOTAL_LEN, stride):
        prompts.append(all_ids[i : i + PROMPT_LEN])
        targets.append(all_ids[i + PROMPT_LEN : i + TOTAL_LEN])
        if len(prompts) >= max_samples:
            break

    return torch.tensor(prompts, dtype=torch.long), torch.tensor(targets, dtype=torch.long)

train_p, train_t = tokenize_and_chunk(train_texts, max_samples=8000)
val_p,   val_t   = tokenize_and_chunk(val_texts,   max_samples=1000)

class TextPairDataset(Dataset):
    def __init__(self, prompts, targets):
        self.prompts = prompts
        self.targets = targets
    def __len__(self):
        return len(self.prompts)
    def __getitem__(self, idx):
        return self.prompts[idx], self.targets[idx]

BATCH_SIZE = 8
train_loader = DataLoader(TextPairDataset(train_p, train_t), batch_size=BATCH_SIZE, shuffle=True,  drop_last=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(TextPairDataset(val_p,   val_t),   batch_size=BATCH_SIZE, shuffle=False, drop_last=True, num_workers=2, pin_memory=True)

print(f"\nDataset procesado:")
print(f"  Train : {len(train_p):,} samples × ({PROMPT_LEN} prompt + {SEQ_LEN} target) tokens")
print(f"  Val   : {len(val_p):,} samples × ({PROMPT_LEN} prompt + {SEQ_LEN} target) tokens")
print(f"  Batch : {BATCH_SIZE}")

In [ ]:
# ── Cell 5: Instanciar modelo (~41M params) & Multi-GPU ────────────────
print("=" * 75)
print("🏗️  CONSTRUYENDO MODELO LinearBlockDiffusion")
print("=" * 75)

D_MODEL             = 512
NUM_LAYERS          = 6
BLOCK_SIZE          = 512
OVERLAP_RATIO       = 0.5
NUM_DIFFUSION_STEPS = 8
CHUNK_DENOISE_SIZE  = 64

raw_model = LinearBlockDiffusionArchitecture(
    vocab_size          = VOCAB_SIZE,
    d_model             = D_MODEL,
    num_layers          = NUM_LAYERS,
    block_size          = BLOCK_SIZE,
    overlap_ratio       = OVERLAP_RATIO,
    num_diffusion_steps = NUM_DIFFUSION_STEPS,
    chunk_denoise_size  = CHUNK_DENOISE_SIZE,
    pad_token_id        = tokenizer.eos_token_id,
    eos_token_id        = tokenizer.eos_token_id,
    noise_injection_prob = 0.15,
    dropout             = 0.1,
)

num_params = sum(p.numel() for p in raw_model.parameters() if p.requires_grad)
print(f"Parámetros entrenables: {num_params / 1e6:.2f}M (Target: 20M–50M)")

raw_model = raw_model.to(DEVICE)

if NUM_GPUS > 1:
    print(f"🔗 DataParallel activado: {NUM_GPUS} GPUs")
    model = nn.DataParallel(raw_model)
else:
    model = raw_model

print(f"Modelo en: {DEVICE}")

In [ ]:
# ── Cell 6: Training Loop Optimizado (Métricas en tiempo real) ─────────
print("=" * 75)
print("⚡ ENTRENAMIENTO — MIDIENDO TOKENS/S, VRAM, MS/PASO")
print("=" * 75)

EPOCHS = 3
LR = 5e-4
MAX_STEPS_PER_EPOCH = 200

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01, betas=(0.9, 0.95))
total_steps = EPOCHS * min(MAX_STEPS_PER_EPOCH, len(train_loader))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-5)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

total_tokens_trained = 0
total_train_time = 0.0
step_count = 0
train_losses = []
peak_vram_train = 0.0

model.train()
for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    epoch_steps = 0
    print(f"\n{'─'*75}")
    print(f"  EPOCH {epoch}/{EPOCHS}")
    print(f"{'─'*75}")

    for step, (p_batch, t_batch) in enumerate(train_loader):
        if step >= MAX_STEPS_PER_EPOCH:
            break

        t0 = time.perf_counter()
        p_batch = p_batch.to(DEVICE, non_blocking=True)
        t_batch = t_batch.to(DEVICE, non_blocking=True)
        batch_tokens = t_batch.numel()

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            # return_logits=False evita transferir 1.6 GB de logits por PCIe cada paso
            loss = model(p_batch, target_ids=t_batch, return_logits=False)
            if isinstance(model, nn.DataParallel):
                loss = loss.mean()

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        dt = time.perf_counter() - t0
        total_train_time += dt
        total_tokens_trained += batch_tokens
        step_count += 1
        epoch_loss += loss.item()
        epoch_steps += 1
        train_losses.append(loss.item())

        if torch.cuda.is_available():
            vram_gb = torch.cuda.max_memory_allocated() / 1e9
            peak_vram_train = max(peak_vram_train, vram_gb)

        tok_s = batch_tokens / dt
        ms_step = dt * 1000

        if (step + 1) % 25 == 0 or step == 0:
            ppl = math.exp(min(loss.item(), 20.0))
            print(f"  [{step+1:3d}/{MAX_STEPS_PER_EPOCH}] "
                  f"Loss: {loss.item():.4f} | "
                  f"PPL: {ppl:8.2f} | "
                  f"{ms_step:6.1f} ms/step | "
                  f"{tok_s:8.0f} tok/s | "
                  f"VRAM: {peak_vram_train:.2f} GB")

    avg_epoch_loss = epoch_loss / max(epoch_steps, 1)
    print(f"  → Epoch {epoch} avg loss: {avg_epoch_loss:.4f} | PPL: {math.exp(min(avg_epoch_loss, 20)):.2f}")

avg_train_tok_s = total_tokens_trained / total_train_time
avg_ms_per_step = (total_train_time / step_count) * 1000
print(f"\n{'='*75}")
print(f"✅ Entrenamiento completado: {step_count} pasos en {total_train_time:.1f}s")
print(f"   Promedio: {avg_train_tok_s:,.0f} tok/s | {avg_ms_per_step:.1f} ms/paso | Peak VRAM: {peak_vram_train:.2f} GB")

In [ ]:
# ── Cell 7: Evaluación de Validación & Perplejidad ─────────────────────
print("=" * 75)
print("📉 EVALUACIÓN — VALIDATION LOSS & PERPLEJIDAD")
print("=" * 75)

model.eval()
val_loss_sum = 0.0
val_steps = 0
MAX_VAL_STEPS = 50

with torch.no_grad():
    for p_batch, t_batch in val_loader:
        if val_steps >= MAX_VAL_STEPS:
            break
        p_batch = p_batch.to(DEVICE, non_blocking=True)
        t_batch = t_batch.to(DEVICE, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            loss = model(p_batch, target_ids=t_batch, return_logits=False)
            if isinstance(model, nn.DataParallel):
                loss = loss.mean()

        val_loss_sum += loss.item()
        val_steps += 1

avg_val_loss = val_loss_sum / max(val_steps, 1)
val_ppl = math.exp(min(avg_val_loss, 20.0))

print(f"  Validation Loss : {avg_val_loss:.4f}")
print(f"  Validation PPL  : {val_ppl:.2f}")
print(f"  Pasos evaluados : {val_steps}")

In [ ]:
# ── Cell 8: Benchmark de Inferencia & Generación de Texto ──────────────
print("=" * 75)
print("🎯 INFERENCIA — BENCHMARK & GENERACIÓN DE TEXTO")
print("=" * 75)

eval_model = raw_model
eval_model.eval()

test_prompts = [
    "In a distant world, scientists discovered",
    "The history of artificial intelligence began",
    "Once upon a time in a small village",
]

MAX_NEW_TOKENS = 512
all_infer_times = []
all_infer_tokens = []

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

for i, prompt_text in enumerate(test_prompts):
    prompt_ids = tokenizer.encode(prompt_text, return_tensors="pt").to(DEVICE)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    generated_ids = eval_model.generate(
        prompt_ids,
        max_new_tokens=MAX_NEW_TOKENS,
        block_size=512,
        overlap_ratio=0.5,
        num_diffusion_steps=8,
        chunk_denoise_size=64,
        temperature=0.8,
        top_k=40,
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    dt = time.perf_counter() - t0

    gen_count = generated_ids.shape[1] - prompt_ids.shape[1]
    all_infer_times.append(dt)
    all_infer_tokens.append(gen_count)

    decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    print(f"\n{'─'*75}")
    print(f"  Sample {i+1} | Prompt: \"{prompt_text}\"")
    print(f"  Tokens generados: {gen_count} | Tiempo: {dt:.2f}s | {gen_count/dt:.1f} tok/s")
    print(f"{'─'*75}")
    print(decoded[:800])

total_infer_tokens = sum(all_infer_tokens)
total_infer_time = sum(all_infer_times)
avg_infer_tok_s = total_infer_tokens / total_infer_time
total_blocks = sum(math.ceil(t / 256) for t in all_infer_tokens)
avg_ms_per_block = (total_infer_time / total_blocks) * 1000
infer_peak_vram = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0

print(f"\n{'='*75}")
print(f"📊 Inferencia promedio: {avg_infer_tok_s:.1f} tok/s | {avg_ms_per_block:.1f} ms/bloque | Peak VRAM: {infer_peak_vram:.2f} GB")

In [ ]:
# ── Cell 9: Curva de Loss durante entrenamiento ────────────────────────
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, linewidth=0.8, alpha=0.5, label="Loss por paso")
window = 20
if len(train_losses) >= window:
    ma = [sum(train_losses[i:i+window])/window for i in range(len(train_losses)-window+1)]
    ax1.plot(range(window-1, len(train_losses)), ma, linewidth=2, color='red', label=f"Media móvil ({window})")
ax1.axhline(avg_val_loss, color='orange', linestyle='--', linewidth=1.5, label=f"Val Loss: {avg_val_loss:.4f}")
ax1.set_xlabel("Paso")
ax1.set_ylabel("Loss")
ax1.set_title("📉 Training Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

train_ppls = [math.exp(min(l, 15)) for l in train_losses]
ax2.plot(train_ppls, linewidth=0.8, alpha=0.5, color='green', label="PPL por paso")
if len(train_ppls) >= window:
    ma_ppl = [sum(train_ppls[i:i+window])/window for i in range(len(train_ppls)-window+1)]
    ax2.plot(range(window-1, len(train_ppls)), ma_ppl, linewidth=2, color='darkgreen', label=f"Media móvil ({window})")
ax2.axhline(val_ppl, color='orange', linestyle='--', linewidth=1.5, label=f"Val PPL: {val_ppl:.2f}")
ax2.set_xlabel("Paso")
ax2.set_ylabel("Perplejidad")
ax2.set_title("📊 Training Perplexity")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("LinearBlockDiffusion — Curvas de Entrenamiento", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ Curvas de entrenamiento generadas.")

In [ ]:
# ── Cell 10: Tabla de Benchmark Final ──────────────────────────────────
print("=" * 75)
print("🏆 REPORTE FINAL — LINEAR BLOCK DIFFUSION")
print("=" * 75)

df_config = pd.DataFrame({
    "Parámetro": [
        "Parámetros entrenables", "d_model", "Capas del Refiner",
        "Vocab size (GPT-2)", "Block size", "Overlap ratio",
        "Pasos de difusión (K)", "Chunk denoise size", "Noise injection prob",
        "Épocas entrenadas", "Pasos totales", "Batch size", "Learning rate",
    ],
    "Valor": [
        f"{num_params/1e6:.2f}M", str(D_MODEL), str(NUM_LAYERS),
        f"{VOCAB_SIZE:,}", str(BLOCK_SIZE), str(OVERLAP_RATIO),
        str(NUM_DIFFUSION_STEPS), str(CHUNK_DENOISE_SIZE), "0.15",
        str(EPOCHS), str(step_count), str(BATCH_SIZE), str(LR),
    ]
})
print("\n📋 Configuración del Modelo")
display(df_config)

df_perf = pd.DataFrame({
    "Métrica": [
        "🏋️ Velocidad Entrenamiento",
        "🏋️ Tiempo por paso (entrenamiento)",
        "🏋️ Peak VRAM (entrenamiento)",
        "🏋️ Loss final (train)",
        "🏋️ PPL final (train)",
        "📉 Validation Loss",
        "📉 Validation PPL",
        "🚀 Velocidad Inferencia",
        "🚀 Tiempo por bloque (inferencia)",
        "🚀 Peak VRAM (inferencia)",
    ],
    "Valor": [
        f"{avg_train_tok_s:,.0f} tok/s",
        f"{avg_ms_per_step:.1f} ms/paso",
        f"{peak_vram_train:.2f} GB",
        f"{train_losses[-1]:.4f}",
        f"{math.exp(min(train_losses[-1], 20)):.2f}",
        f"{avg_val_loss:.4f}",
        f"{val_ppl:.2f}",
        f"{avg_infer_tok_s:.1f} tok/s",
        f"{avg_ms_per_block:.1f} ms/bloque",
        f"{infer_peak_vram:.2f} GB",
    ]
})
print("\n📊 Métricas de Rendimiento")
display(df_perf)

df_samples = pd.DataFrame({
    "Muestra": [f"Sample {i+1}" for i in range(len(test_prompts))],
    "Prompt": [p[:40] + "..." for p in test_prompts],
    "Tokens generados": all_infer_tokens,
    "Tiempo (s)": [f"{t:.2f}" for t in all_infer_times],
    "Tokens/s": [f"{t/d:.1f}" for t, d in zip(all_infer_tokens, all_infer_times)],
})
print("\n🎯 Detalle de Inferencia por Muestra")
display(df_samples)

print(f"\n{'='*75}")
print("✅ Benchmark completado exitosamente.")
print(f"{'='*75}")